# Caderno 14 - Compila métricas para expansão de queries

As tabelas são as combinações dos conjuntos de queries (3 conjuntos) e k = [5, 10, 20].

## 1. Pastas com os resultados, nomes dos arquivos de índice. docs/queries/qrels

In [1]:
import pandas as pd

PASTA_DADOS = './dados/'
# A pasta dos JURIS aqui não é a pasta original, e sim o resultado do caderno 1 (os documentos já estão filtrados)
PASTA_JURIS_TCU = f'{PASTA_DADOS}outputs/1_tratamento_juris_tcu/'

# Cenário BM25 padrão, com enunciado único e com enunciado dobrado
INDICE_BM25_PADRAO = f'{PASTA_DADOS}/outputs/4_metricas_bm25_padrao/indice_js_enunciado_e_excerto.pickle'
# Cenário BM25 padrão com DOC2QUERY (5 queries) e sinônimos do enunciado
INDICE_BM25_COM_DOC2QUERY_5QUERIES_E_SINONIMOS_ENUNCIADO_GPT = f'{PASTA_DADOS}/outputs/11_metricas_bm25_com_doc2query_e_sinonimos_enunciado_gpt_llama/indice_js_enunciado_e_excerto_e_doc2query_5q_e_gpt.pickle'
INDICE_BM25_COM_DOC2QUERY_5QUERIES_E_SINONIMOS_ENUNCIADO_GPT_4o = f'{PASTA_DADOS}/outputs/11_metricas_bm25_com_doc2query_e_sinonimos_enunciado_gpt_llama/indice_js_enunciado_e_excerto_e_doc2query_5q_e_gpt4o.pickle'
INDICE_BM25_COM_DOC2QUERY_5QUERIES_E_SINONIMOS_ENUNCIADO_LLAMA = f'{PASTA_DADOS}/outputs/11_metricas_bm25_com_doc2query_e_sinonimos_enunciado_gpt_llama/indice_js_enunciado_e_excerto_e_doc2query_5q_e_llama.pickle'

In [2]:
# Carrega os arquivos 
def carrega_juris_tcu():
    doc1 = pd.read_csv(f'{PASTA_JURIS_TCU}doc_tratado_parte_1.csv', sep='|')
    doc2 = pd.read_csv(f'{PASTA_JURIS_TCU}doc_tratado_parte_2.csv', sep='|')
    doc3 = pd.read_csv(f'{PASTA_JURIS_TCU}doc_tratado_parte_3.csv', sep='|')
    doc4 = pd.read_csv(f'{PASTA_JURIS_TCU}doc_tratado_parte_4.csv', sep='|')
    doc = pd.concat([doc1, doc2, doc3, doc4], ignore_index=True)
    query = pd.read_csv(f'{PASTA_JURIS_TCU}query_tratado.csv', sep='|')
    qrel = pd.read_csv(f'{PASTA_JURIS_TCU}qrel_tratado.csv', sep='|')

    return doc, query, qrel

docs, queries, qrels = carrega_juris_tcu()

In [3]:
# Carrega expansão de queries
import pickle

NOME_ARQUIVO_RESULTADO_GPT35 = f'{PASTA_DADOS}outputs/13_expansao_queries_com_gpt_llama/expansao_queries_gpt35.pickle'
NOME_ARQUIVO_RESULTADO_GPT4O = f'{PASTA_DADOS}outputs/13_expansao_queries_com_gpt_llama/expansao_queries_gpt4o.pickle'
NOME_ARQUIVO_RESULTADO_LLAMA = f'{PASTA_DADOS}outputs/13_expansao_queries_com_gpt_llama/expansao_queries_llama.pickle'

# Objetos com a expansão das queries
expansao_queries_gpt35 = {}
expansao_queries_gpt4o = {}
expansao_queries_llama = {}

with open(NOME_ARQUIVO_RESULTADO_GPT35, 'rb') as f:
    expansao_queries_gpt35 = pickle.load(f)
print("Resultados do GPT-3.5 recuperados")

with open(NOME_ARQUIVO_RESULTADO_GPT4O, 'rb') as f:
    expansao_queries_gpt4o = pickle.load(f)
print("Resultados do GPT-4o recuperados")

# Verifica se os arquivos já existem. Se já existem, recupera.
with open(NOME_ARQUIVO_RESULTADO_LLAMA, 'rb') as f:
    expansao_queries_llama = pickle.load(f)
    print("Resultados do Llama recuperados")

Resultados do GPT-3.5 recuperados
Resultados do GPT-4o recuperados
Resultados do Llama recuperados


In [4]:
# Cria colunas contendo a expansão das query
queries['TEXT-EXPANSAO-GPT-3.5'] = queries['TEXT'] + ' ' + queries['KEY'].map(expansao_queries_gpt35)
queries['TEXT-EXPANSAO-GPT-4o'] = queries['TEXT'] + ' ' + queries['KEY'].map(expansao_queries_gpt4o)
queries['TEXT-EXPANSAO-Llama-3'] = queries['TEXT'] + ' ' + queries['KEY'].map(expansao_queries_llama)

## 2. Cria índices e buscadores

In [5]:
from bm25 import IndiceInvertido, BM25, tokenizador_pt_remove_html

# Lista ordenada para exibição
# Apenas inverte a visualização da reescrita com os sinônimos
lista_nome_buscadores = ["bm25_padrao",

                       "expansao_queries_gpt_3.5",
                       "expansao_queries_gpt_4o",
                       "expansao_queries_llama-3",

                       "expansao_queries_doc2query_5queries_sinonimos_gpt_3.5",
                       "expansao_queries_doc2query_5queries_sinonimos_gpt_4o",
                       "expansao_queries_doc2query_5queries_sinonimos_llama-3"]

mapa_nome_arquivos_indices = {
    "bm25_padrao": INDICE_BM25_PADRAO,

    "expansao_queries_gpt_3.5": INDICE_BM25_PADRAO,
    "expansao_queries_gpt_4o": INDICE_BM25_PADRAO,
    "expansao_queries_llama-3": INDICE_BM25_PADRAO,
    
    "expansao_queries_doc2query_5queries_sinonimos_gpt_3.5": INDICE_BM25_COM_DOC2QUERY_5QUERIES_E_SINONIMOS_ENUNCIADO_GPT,
    "expansao_queries_doc2query_5queries_sinonimos_gpt_4o": INDICE_BM25_COM_DOC2QUERY_5QUERIES_E_SINONIMOS_ENUNCIADO_GPT_4o,
    "expansao_queries_doc2query_5queries_sinonimos_llama-3": INDICE_BM25_COM_DOC2QUERY_5QUERIES_E_SINONIMOS_ENUNCIADO_LLAMA
}

mapa_nome_coluna_queries = {
    "bm25_padrao": 'TEXT',

    "expansao_queries_gpt_3.5": 'TEXT-EXPANSAO-GPT-3.5',
    "expansao_queries_gpt_4o": 'TEXT-EXPANSAO-GPT-4o',
    "expansao_queries_llama-3": 'TEXT-EXPANSAO-Llama-3',
    
    "expansao_queries_doc2query_5queries_sinonimos_gpt_3.5": 'TEXT-EXPANSAO-GPT-3.5',
    "expansao_queries_doc2query_5queries_sinonimos_gpt_4o": 'TEXT-EXPANSAO-GPT-4o',
    "expansao_queries_doc2query_5queries_sinonimos_llama-3": 'TEXT-EXPANSAO-Llama-3'
}


# Cria os IndiceInvertidos
mapa_indices = {nome: IndiceInvertido(tokenizador_pt_remove_html) for nome, _ in mapa_nome_arquivos_indices.items()}

# Carrega dos pickles
for nome, indice in mapa_indices.items():
    print(f'Carregando indice {nome} com o arquivo {mapa_nome_arquivos_indices[nome]}')
    indice.from_pickle(mapa_nome_arquivos_indices[nome])

# Cria os buscadores
# Sobre a escolha de k1 e b:
# https://www.elastic.co/pt/blog/practical-bm25-part-3-considerations-for-picking-b-and-k1-in-elasticsearch
mapa_buscadores = {nome: BM25(indice, k1=1.2, b=0.75, bias_idf=1) for nome, indice in mapa_indices.items()}

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\P_8454\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\P_8454\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package rslp to
[nltk_data]     C:\Users\P_8454\AppData\Roaming\nltk_data...
[nltk_data]   Package rslp is already up-to-date!


Carregando indice bm25_padrao com o arquivo ./dados//outputs/4_metricas_bm25_padrao/indice_js_enunciado_e_excerto.pickle
Carregando indice expansao_queries_gpt_3.5 com o arquivo ./dados//outputs/4_metricas_bm25_padrao/indice_js_enunciado_e_excerto.pickle
Carregando indice expansao_queries_gpt_4o com o arquivo ./dados//outputs/4_metricas_bm25_padrao/indice_js_enunciado_e_excerto.pickle
Carregando indice expansao_queries_llama-3 com o arquivo ./dados//outputs/4_metricas_bm25_padrao/indice_js_enunciado_e_excerto.pickle
Carregando indice expansao_queries_doc2query_5queries_sinonimos_gpt_3.5 com o arquivo ./dados//outputs/11_metricas_bm25_com_doc2query_e_sinonimos_enunciado_gpt_llama/indice_js_enunciado_e_excerto_e_doc2query_5q_e_gpt.pickle
Carregando indice expansao_queries_doc2query_5queries_sinonimos_gpt_4o com o arquivo ./dados//outputs/11_metricas_bm25_com_doc2query_e_sinonimos_enunciado_gpt_llama/indice_js_enunciado_e_excerto_e_doc2query_5q_e_gpt4o.pickle
Carregando indice expansao_qu

## 3. Extrai as métricas

In [6]:
from tqdm import tqdm

def pesquisar_usando_buscador(nome, buscador, nome_coluna_query):
    col_resultado_query_key=[]
    col_resultado_doc_key=[]
    col_resultado_rank=[]
    
    for i, row in tqdm(queries.iterrows(), total=len(queries), desc=f'Pesquisando usando {nome}. Query: {nome_coluna_query}'):
        query_key = row.KEY
        query_text = row[nome_coluna_query]
        resultados = buscador.pesquisar(query_text)
    
        primeiros_50_docs = [tupla_key_score[0] for tupla_key_score in resultados[:50]]
        queries_keys = [query_key] * len(primeiros_50_docs)
        ranking = list(range(1, len(primeiros_50_docs)+1))
    
        col_resultado_query_key.extend(queries_keys)
        col_resultado_doc_key.extend(primeiros_50_docs)
        col_resultado_rank.extend(ranking)
    
    df_resultados = pd.DataFrame({
        "QUERY_KEY": col_resultado_query_key,
        "DOC_KEY": col_resultado_doc_key,
        "RANK": col_resultado_rank,
    })
    return df_resultados

In [7]:
from metricas import metricas

# Pesquisa em todos os buscadores
mapa_resultados = {nome: pesquisar_usando_buscador(nome, buscador, mapa_nome_coluna_queries[nome]) for nome, buscador in mapa_buscadores.items()}

# Extrai as métricas
mapa_metricas = {nome: metricas(resultados, qrels, aproximacao_trec_eval=True) for nome, resultados in mapa_resultados.items()}

Pesquisando usando bm25_padrao. Query: TEXT: 100%|██████████████████████████████████| 150/150 [00:01<00:00, 101.10it/s]
Pesquisando usando expansao_queries_gpt_3.5. Query: TEXT-EXPANSAO-GPT-3.5: 100%|█████| 150/150 [00:02<00:00, 66.90it/s]
Pesquisando usando expansao_queries_gpt_4o. Query: TEXT-EXPANSAO-GPT-4o: 100%|███████| 150/150 [00:02<00:00, 65.13it/s]
Pesquisando usando expansao_queries_llama-3. Query: TEXT-EXPANSAO-Llama-3: 100%|█████| 150/150 [00:02<00:00, 63.88it/s]
Pesquisando usando expansao_queries_doc2query_5queries_sinonimos_gpt_3.5. Query: TEXT-EXPANSAO-GPT-3.5: 100%|█| 150/150
Pesquisando usando expansao_queries_doc2query_5queries_sinonimos_gpt_4o. Query: TEXT-EXPANSAO-GPT-4o: 100%|█| 150/150 [
Pesquisando usando expansao_queries_doc2query_5queries_sinonimos_llama-3. Query: TEXT-EXPANSAO-Llama-3: 100%|█| 150/150


## 4. Exibe as métricas

In [8]:
# Imprime as métricas para o conjunto de queries 1 (0:50), 2 (100:150), ou 3 (100:150) 
# e para um determinado k (foi gerado para k = 5, 10, 20 e 50).

def compara_metricas(con_query, k):
    # Acumula as métricas
    precisao = []
    recall = []
    mrr = []
    ndcg = []

    for nome in lista_nome_buscadores:
        estatisticas = mapa_metricas[nome][50*(con_query-1):50*(con_query)].describe()
        precisao.append(estatisticas.loc['mean', f'P@{k}'])
        recall.append(estatisticas.loc['mean', f'R@{k}'])
        mrr.append(estatisticas.loc['mean', f'MRR@{k}'])
        ndcg.append(estatisticas.loc['mean', f'nDCG@{k}'])

    df = pd.DataFrame({
        "Modelo": lista_nome_buscadores,
        f"P@{k}": precisao,
        f"R@{k}": recall,
        f"MRR@{k}": mrr,
        f"nDCG@{k}": ndcg
    })
    return df

def compara_metricas_todas_queries(k):
    # Acumula as métricas
    precisao = []
    recall = []
    mrr = []
    ndcg = []

    for nome in lista_nome_buscadores:
        estatisticas = mapa_metricas[nome].describe()
        precisao.append(estatisticas.loc['mean', f'P@{k}'])
        recall.append(estatisticas.loc['mean', f'R@{k}'])
        mrr.append(estatisticas.loc['mean', f'MRR@{k}'])
        ndcg.append(estatisticas.loc['mean', f'nDCG@{k}'])

    df = pd.DataFrame({
        "Modelo": lista_nome_buscadores,
        f"P@{k}": precisao,
        f"R@{k}": recall,
        f"MRR@{k}": mrr,
        f"nDCG@{k}": ndcg
    })
    return df

pd.set_option('display.precision', 4)

In [13]:
for con_query in [1, 2, 3]:
#    for k in [5, 10, 20]:
    for k in [5]:
        print(f'Resultados para conjunto de query {con_query} e k={k}')
        display(compara_metricas(con_query, k))

Resultados para conjunto de query 1 e k=5


,Modelo,P@5,R@5,MRR@5,nDCG@5
0,bm25_padrao,0.272,0.1106,0.5253,0.2824
1,expansao_queries_gpt_3.5,0.140,0.0594,0.2507,0.1359
2,expansao_queries_gpt_4o,0.136,0.0572,0.2947,0.1447
3,expansao_queries_llama-3,0.168,0.0721,0.3590,0.1799
4,expansao_queries_doc2query_5queries_sinonimos_...,0.240,0.0991,0.4280,0.2367
5,expansao_queries_doc2query_5queries_sinonimos_...,0.220,0.0903,0.3870,0.2128
6,expansao_queries_doc2query_5queries_sinonimos_...,0.276,0.1168,0.4567,0.2653


Resultados para conjunto de query 2 e k=5


,Modelo,P@5,R@5,MRR@5,nDCG@5
0,bm25_padrao,0.500,0.2077,0.8620,0.5713
1,expansao_queries_gpt_3.5,0.400,0.1696,0.6523,0.4308
2,expansao_queries_gpt_4o,0.348,0.1489,0.6597,0.3827
3,expansao_queries_llama-3,0.380,0.1605,0.6983,0.4123
4,expansao_queries_doc2query_5queries_sinonimos_...,0.472,0.2003,0.7323,0.5095
5,expansao_queries_doc2query_5queries_sinonimos_...,0.468,0.1968,0.7667,0.5012
6,expansao_queries_doc2query_5queries_sinonimos_...,0.484,0.2016,0.7477,0.5152


Resultados para conjunto de query 3 e k=5


,Modelo,P@5,R@5,MRR@5,nDCG@5
0,bm25_padrao,0.520,0.2340,0.9150,0.6030
1,expansao_queries_gpt_3.5,0.472,0.2095,0.8233,0.5493
2,expansao_queries_gpt_4o,0.444,0.1998,0.7763,0.5068
3,expansao_queries_llama-3,0.476,0.2143,0.8200,0.5334
4,expansao_queries_doc2query_5queries_sinonimos_...,0.544,0.2422,0.9167,0.6230
5,expansao_queries_doc2query_5queries_sinonimos_...,0.496,0.2218,0.7980,0.5508
6,expansao_queries_doc2query_5queries_sinonimos_...,0.492,0.2206,0.8317,0.5583


In [10]:
for con_query in [1, 2, 3]:
#    for k in [5, 10, 20]:
    for k in [10]:
        print(f'Resultados para conjunto de query {con_query} e k={k}')
        display(compara_metricas(con_query, k))

Resultados para conjunto de query 1 e k=10


,Modelo,P@10,R@10,MRR@10,nDCG@10
0,bm25_padrao,0.238,0.1966,0.5386,0.2753
1,expansao_queries_gpt_3.5,0.126,0.1039,0.2780,0.1367
2,expansao_queries_gpt_4o,0.126,0.1065,0.3127,0.1405
3,expansao_queries_llama-3,0.146,0.1227,0.3866,0.1728
4,expansao_queries_doc2query_5queries_sinonimos_...,0.196,0.1605,0.4474,0.2202
5,expansao_queries_doc2query_5queries_sinonimos_...,0.190,0.1553,0.4052,0.2029
6,expansao_queries_doc2query_5queries_sinonimos_...,0.230,0.1929,0.4717,0.2535


Resultados para conjunto de query 2 e k=10


,Modelo,P@10,R@10,MRR@10,nDCG@10
0,bm25_padrao,0.378,0.3176,0.8665,0.5106
1,expansao_queries_gpt_3.5,0.326,0.2742,0.6680,0.4096
2,expansao_queries_gpt_4o,0.296,0.2474,0.6698,0.3700
3,expansao_queries_llama-3,0.314,0.2638,0.7067,0.3942
4,expansao_queries_doc2query_5queries_sinonimos_...,0.372,0.3118,0.7432,0.4723
5,expansao_queries_doc2query_5queries_sinonimos_...,0.346,0.2894,0.7725,0.4493
6,expansao_queries_doc2query_5queries_sinonimos_...,0.360,0.3008,0.7539,0.4619


Resultados para conjunto de query 3 e k=10


,Modelo,P@10,R@10,MRR@10,nDCG@10
0,bm25_padrao,0.388,0.3451,0.9175,0.5328
1,expansao_queries_gpt_3.5,0.362,0.3219,0.8262,0.4916
2,expansao_queries_gpt_4o,0.350,0.3117,0.7855,0.4642
3,expansao_queries_llama-3,0.384,0.3402,0.8262,0.4921
4,expansao_queries_doc2query_5queries_sinonimos_...,0.402,0.3563,0.9187,0.5446
5,expansao_queries_doc2query_5queries_sinonimos_...,0.390,0.3478,0.8000,0.5029
6,expansao_queries_doc2query_5queries_sinonimos_...,0.430,0.3796,0.8367,0.5342


In [11]:
for k in [5, 10, 20]:
    display(compara_metricas_todas_queries(k))

,Modelo,P@5,R@5,MRR@5,nDCG@5
0,bm25_padrao,0.4307,0.1841,0.7674,0.4856
1,expansao_queries_gpt_3.5,0.3373,0.1462,0.5754,0.3720
2,expansao_queries_gpt_4o,0.3093,0.1353,0.5769,0.3447
3,expansao_queries_llama-3,0.3413,0.1489,0.6258,0.3752
4,expansao_queries_doc2query_5queries_sinonimos_...,0.4187,0.1805,0.6923,0.4564
5,expansao_queries_doc2query_5queries_sinonimos_...,0.3947,0.1696,0.6506,0.4216
6,expansao_queries_doc2query_5queries_sinonimos_...,0.4173,0.1797,0.6787,0.4463


,Modelo,P@10,R@10,MRR@10,nDCG@10
0,bm25_padrao,0.3347,0.2864,0.7742,0.4396
1,expansao_queries_gpt_3.5,0.2713,0.2333,0.5907,0.3460
2,expansao_queries_gpt_4o,0.2573,0.2219,0.5893,0.3249
3,expansao_queries_llama-3,0.2813,0.2423,0.6398,0.3530
4,expansao_queries_doc2query_5queries_sinonimos_...,0.3233,0.2762,0.7031,0.4124
5,expansao_queries_doc2query_5queries_sinonimos_...,0.3087,0.2642,0.6592,0.3850
6,expansao_queries_doc2query_5queries_sinonimos_...,0.3400,0.2911,0.6874,0.4165


,Modelo,P@20,R@20,MRR@20,nDCG@20
0,bm25_padrao,0.2497,0.4258,0.7762,0.5004
1,expansao_queries_gpt_3.5,0.1977,0.3365,0.5974,0.3914
2,expansao_queries_gpt_4o,0.1927,0.3300,0.5933,0.3752
3,expansao_queries_llama-3,0.2050,0.3504,0.6425,0.4029
4,expansao_queries_doc2query_5queries_sinonimos_...,0.2297,0.3899,0.7061,0.4600
5,expansao_queries_doc2query_5queries_sinonimos_...,0.2330,0.3967,0.6619,0.4455
6,expansao_queries_doc2query_5queries_sinonimos_...,0.2353,0.4002,0.6892,0.4634
